In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución de población por sexo y sección censal

Se han estructurado los datos en la carpeta de inputs para la dimensión demográfica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del censo anual de población
https://www.ine.es/dynt3/inebase/index.htm?padre=11555&capsel=11100

In [2]:
path = os.path.join(DATA_INPUTS_DD, "Poblacion por sexo y edad")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 936408 entries, 528 to 1532255
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Provincias  936408 non-null  object 
 1   Municipios  936408 non-null  object 
 2   Secciones   936408 non-null  object 
 3   Sexo        936408 non-null  object 
 4   Edad        936408 non-null  object 
 5   Periodo     936408 non-null  int64  
 6   Total       931854 non-null  float64
 7   Provincia   936408 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 64.3+ MB
None


,Provincias,Municipios,Secciones,Sexo,Edad,Periodo,Total,Provincia
1213299,47 Valladolid,47066 Fuensaldaña,4706601001 Fuensaldaña sección 01001,Mujeres,De 45 a 49 años,2021,71.0,Valladolid
1471627,49 Zamora,49175 Revellinos,4917501001 Revellinos sección 01001,Hombres,Todas las edades,2021,136.0,Zamora
379204,09 Burgos,09434 Villagonzalo Pedernales,0943401001 Villagonzalo Pedernales sección 01001,Hombres,De 10 a 14 años,2024,88.0,Burgos
1169109,42 Soria,42213 Villaseca de Arciel,4221301001 Villaseca de Arciel sección 01001,Hombres,De 30 a 34 años,2023,1.0,Soria
203020,09 Burgos,09059 Burgos,0905907002 Burgos sección 07002,Total,De 0 a 4 años,2024,30.0,Burgos


# Filtrado
No interesan realmente todos los grupos quinquenales, por lo que se puede filtrar por "Todas las edades" y tal que el Sexo sea 
distinto del total, para ver la distribución

In [3]:
indicadores_filtrado = indicadores[
    (indicadores["Edad"] == "Todas las edades") &
    (indicadores["Sexo"] != "Total")
].copy()

# Veo una muestra de su estructura y contenido
print(indicadores_filtrado.info())
indicadores_filtrado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 28376 entries, 616 to 1532171
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Provincias  28376 non-null  object 
 1   Municipios  28376 non-null  object 
 2   Secciones   28376 non-null  object 
 3   Sexo        28376 non-null  object 
 4   Edad        28376 non-null  object 
 5   Periodo     28376 non-null  int64  
 6   Total       28238 non-null  float64
 7   Provincia   28376 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1.9+ MB
None


,Provincias,Municipios,Secciones,Sexo,Edad,Periodo,Total,Provincia
1354408,47 Valladolid,47186 Valladolid,4718611034 Valladolid sección 11034,Hombres,Todas las edades,2024,743.0,Valladolid
346283,09 Burgos,09355 Santibáñez de Esgueva,0935501001 Santibáñez de Esgueva sección 01001,Mujeres,Todas las edades,2021,38.0,Burgos
236897,09 Burgos,09100 Cerratón de Juarros,0910001001 Cerratón de Juarros sección 01001,Hombres,Todas las edades,2023,34.0,Burgos
1232795,47 Valladolid,47086 Medina de Rioseco,4708602002 Medina de Rioseco sección 02002,Mujeres,Todas las edades,2021,852.0,Valladolid
157168,09 Burgos,09018 Aranda de Duero,0901801009 Aranda de Duero sección 01009,Hombres,Todas las edades,2024,674.0,Burgos


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [4]:
indicadores_estandarizados = estandarizar_df_ine(indicadores_filtrado, "Sexo")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 28376 entries, 616 to 1532171
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Provincia  28376 non-null  object 
 1   CMuni      28376 non-null  object 
 2   CUSEC      28376 non-null  object 
 3   Indicador  28376 non-null  object 
 4   Periodo    28376 non-null  int64  
 5   Total      28238 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 1.5+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
1354673,Valladolid,47186,4718611035,Hombres,2023,659.0
351561,Burgos,09369,0936901001,Mujeres,2023,229.0
66618,Avila,05113,0511301001,Hombres,2022,32.0
438682,Leon,24061,2406101001,Mujeres,2022,1017.0
1188088,Valladolid,47023,4702301002,Hombres,2024,734.0


In [5]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(indicadores_estandarizados)

📅 Años disponibles:
[2024 2023 2022 2021]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Hombres
  - Mujeres
------------------------------------------------------------


In [6]:
# Modifico el tipo de Total a "Entero"
indicadores_estandarizados["Total"] = pd.to_numeric(
    indicadores_estandarizados["Total"], errors="coerce"
).astype("Int64")  # <-- con mayúscula, permite NaN

In [7]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_estandarizados)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14119 entries, 0 to 14118
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  14119 non-null  object
 1   CMuni      14119 non-null  object
 2   CUSEC      14119 non-null  object
 3   Periodo    14119 non-null  int64 
 4   Hombres    14119 non-null  Int64 
 5   Mujeres    14119 non-null  Int64 
dtypes: Int64(2), int64(1), object(3)
memory usage: 689.5+ KB
None


,Provincia,CMuni,CUSEC,Periodo,Hombres,Mujeres
5317,Leon,24222,2422202002,2021,1105,1183
952,Avila,05196,0519601001,2023,29,33
10201,Soria,42140,4214001001,2022,9,4
8662,Segovia,40016,4001601001,2023,32,23
8889,Segovia,40071,4007101001,2022,22,18


In [8]:
# Aprovecho para añadir una columna con el Total de población, y el % de hombres y mujeres respecto a ese total

# --- Calcular columna total ---
indicadores_por_seccion["Población"] = (
    indicadores_por_seccion["Hombres"].fillna(0) + indicadores_por_seccion["Mujeres"].fillna(0)
).astype("Int64")  # Tipo entero tolerante a nulos

# --- Calcular porcentajes respecto al total ---
indicadores_por_seccion["%_Hombres"] = (
    indicadores_por_seccion["Hombres"] / indicadores_por_seccion["Población"] * 100
).round(2)

indicadores_por_seccion["%_Mujeres"] = (
    indicadores_por_seccion["Mujeres"] / indicadores_por_seccion["Población"] * 100
).round(2)

print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14119 entries, 0 to 14118
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Provincia  14119 non-null  object 
 1   CMuni      14119 non-null  object 
 2   CUSEC      14119 non-null  object 
 3   Periodo    14119 non-null  int64  
 4   Hombres    14119 non-null  Int64  
 5   Mujeres    14119 non-null  Int64  
 6   Población  14119 non-null  Int64  
 7   %_Hombres  14119 non-null  Float64
 8   %_Mujeres  14119 non-null  Float64
dtypes: Float64(2), Int64(3), int64(1), object(3)
memory usage: 1.0+ MB
None


,Provincia,CMuni,CUSEC,Periodo,Hombres,Mujeres,Población,%_Hombres,%_Mujeres
5802,Palencia,34112,3411201001,2024,27,20,47,57.45,42.55
2864,Burgos,09258,0925801001,2021,90,75,165,54.55,45.45
2540,Burgos,09195,0919501001,2023,35,13,48,72.92,27.08
4631,Leon,24115,2411502010,2023,552,641,1193,46.27,53.73
4293,Leon,24089,2408906005,2024,339,432,771,43.97,56.03


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [9]:
# Filtro por los años 2021 - 2023.
indicadores_por_seccion = indicadores_por_seccion[
    indicadores_por_seccion["Periodo"].isin([2021, 2022, 2023])
].copy()

In [10]:
# Aseguramos que las columnas numéricas estén bien convertidas
cols_num = ["Hombres", "Mujeres", "Población", "%_Hombres", "%_Mujeres"]

# --- 1️⃣ NIVEL MUNICIPAL ---
indicadores_por_municipio = (
    indicadores_por_seccion
    .groupby(["Provincia", "CMuni", "Periodo"], as_index=False)
    .agg({
        "Hombres": "sum",
        "Mujeres": "sum",
        "Población": "sum"
    })
)

# Recalculamos los porcentajes ponderados
indicadores_por_municipio["%_Hombres"] = (
    indicadores_por_municipio["Hombres"] / indicadores_por_municipio["Población"]
) * 100

indicadores_por_municipio["%_Mujeres"] = (
    indicadores_por_municipio["Mujeres"] / indicadores_por_municipio["Población"]
) * 100


# --- 2️⃣ NIVEL PROVINCIAL ---
indicadores_por_provincia = (
    indicadores_por_seccion
    .groupby(["Provincia", "Periodo"], as_index=False)
    .agg({
        "Hombres": "sum",
        "Mujeres": "sum",
        "Población": "sum"
    })
)

indicadores_por_provincia["%_Hombres"] = (
    indicadores_por_provincia["Hombres"] / indicadores_por_provincia["Población"]
) * 100

indicadores_por_provincia["%_Mujeres"] = (
    indicadores_por_provincia["Mujeres"] / indicadores_por_provincia["Población"]
) * 100

# Agregado de datos

Aunque para el modelo y el detalle fino se dispondrá de los datos desagregados, es interesante también disponer de los datos
agregados a 2 niveles para la visualizacion:

1. Nivel municipal (por CMuni)
2. Nivel provincial (por Provinincia)


# Export de los resultados

In [11]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DD, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_seccion.csv")
ruta_municipio = os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_municipio.csv")
ruta_provincia = os.path.join(DATA_OUTPUTS_DD, "indicadores_sexo_por_provincia.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_municipio.to_csv(
    ruta_municipio,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_provincia.to_csv(
    ruta_provincia,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
